In [1]:
# Install required packages
!pip install -q transformers datasets torch accelerate scipy nltk

In [2]:
import json
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch import nn
from tqdm.auto import tqdm
import os
import shutil
from typing import List, Dict, Tuple
from scipy.stats import pearsonr
import warnings
import random
import copy
warnings.filterwarnings('ignore')

# Download NLTK data for augmentation
import nltk
try:
    nltk.data.find('wordnet')
except:
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)

In [12]:
# Configuration - OPTIMIZED
class Config:
    # Paths
    TRAIN_ZIP = '/kaggle/input/dimasr/subtask1'
    DATA_DIR = './subtask1_data'
    OUTPUT_DIR = './subtask1_outputs'
    CHECKPOINT_DIR = './subtask1_checkpoints'
    LOG_DIR = './subtask1_logs'
    
    # Model - KEEP BERT (works best for this data size)
    MODEL_NAME = 'bert-base-uncased'
    MAX_LENGTH = 128
    HIDDEN_DIM = 256
    
    # IMPROVED: Stronger regularization
    DROPOUT = 0.4  # Increased from 0.3
    WEIGHT_DECAY = 0.05  # Increased from 0.01
    
    # Training
    BATCH_SIZE = 32
    EPOCHS = 30  # More epochs with early stopping
    LEARNING_RATE = 3e-5  # Slightly higher
    WARMUP_RATIO = 0.1
    GRADIENT_CLIP = 1.0
    
    # IMPROVED: Tighter early stopping
    EARLY_STOPPING_PATIENCE = 3  # Reduced from 5
    
    # Data augmentation
    AUGMENTATION_PROB = 0.4  # 40% of samples augmented
    
    # Ensemble
    ENSEMBLE_SEEDS = [42, 123, 456]  # Train 3 models
    
    # Focal loss
    FOCAL_GAMMA = 2.0  # Focus on hard examples
    FOCAL_WEIGHT = 0.3  # 30% focal, 70% MSE
    
    # VA normalization
    VA_MIN = 1.0
    VA_MAX = 9.0
    
    # Device
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Random seed
    SEED = 42

config = Config()

# Create directories
os.makedirs(config.DATA_DIR, exist_ok=True)
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(config.LOG_DIR, exist_ok=True)

# Copy data files
files_to_copy = [
    "eng_laptop_train_alltasks.jsonl", "eng_laptop_dev_task1.jsonl",
    "eng_laptop_test_task1.jsonl", "eng_restaurant_train_alltasks.jsonl",
    "eng_restaurant_dev_task1.jsonl", "eng_restaurant_test_task1.jsonl"
]

for fname in files_to_copy:
    src = os.path.join(config.TRAIN_ZIP, fname)
    dst = os.path.join(config.DATA_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied {fname}")

print(f"\nDevice: {config.DEVICE}")

Copied eng_laptop_train_alltasks.jsonl
Copied eng_laptop_dev_task1.jsonl
Copied eng_laptop_test_task1.jsonl
Copied eng_restaurant_train_alltasks.jsonl
Copied eng_restaurant_dev_task1.jsonl
Copied eng_restaurant_test_task1.jsonl

Device: cuda


In [13]:
# Data loading functions
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

def normalize_va(value):
    """Normalize VA from [1,9] to [0,1]"""
    return (value - config.VA_MIN) / (config.VA_MAX - config.VA_MIN)

def denormalize_va(value):
    """Denormalize VA from [0,1] to [1,9]"""
    return value * (config.VA_MAX - config.VA_MIN) + config.VA_MIN

def parse_va(va_string):
    """Parse VA string 'V.VV#A.AA' to (valence, arousal)"""
    v, a = va_string.split('#')
    return float(v), float(a)

def format_va(valence, arousal):
    """Format VA to 'V.VV#A.AA' with proper clipping and rounding"""
    v = np.clip(valence, config.VA_MIN, config.VA_MAX)
    a = np.clip(arousal, config.VA_MIN, config.VA_MAX)
    return f"{v:.2f}#{a:.2f}"

In [14]:
# Data Augmentation Functions
from nltk.corpus import wordnet

def get_synonyms(word):
    """Get synonyms for a word"""
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace('_', ' ')
            if synonym.lower() != word.lower():
                synonyms.add(synonym)
    return list(synonyms)

def synonym_replacement(text, aspect, n=2):
    """Replace n words with synonyms (avoid aspect words)"""
    words = text.split()
    aspect_words = set(aspect.lower().split())
    
    # Find replaceable words
    replaceable = []
    for i, word in enumerate(words):
        if word.lower() not in aspect_words and len(word) > 3:
            synonyms = get_synonyms(word)
            if synonyms:
                replaceable.append((i, word, synonyms))
    
    # Replace n random words
    if replaceable:
        n_replace = min(n, len(replaceable))
        to_replace = random.sample(replaceable, n_replace)
        
        for idx, original, synonyms in to_replace:
            words[idx] = random.choice(synonyms)
    
    return ' '.join(words)

def random_insertion(text, n=1):
    """Insert random adjectives/adverbs"""
    modifiers = ['really', 'quite', 'very', 'somewhat', 'rather', 
                 'extremely', 'fairly', 'pretty', 'absolutely']
    
    words = text.split()
    for _ in range(n):
        if len(words) > 2:
            pos = random.randint(1, len(words) - 1)
            words.insert(pos, random.choice(modifiers))
    
    return ' '.join(words)

def augment_text(text, aspect):
    """Apply augmentation techniques"""
    if random.random() < 0.5:
        text = synonym_replacement(text, aspect, n=2)
    
    if random.random() < 0.3:
        text = random_insertion(text, n=1)
    
    return text

def augment_va(v, a, epsilon=0.05):
    """Add small noise to VA (label smoothing)"""
    v_noise = random.uniform(-epsilon, epsilon)
    a_noise = random.uniform(-epsilon, epsilon)
    
    v = np.clip(v + v_noise, 0, 1)
    a = np.clip(a + a_noise, 0, 1)
    
    return v, a

print("✓ Data augmentation functions loaded")

✓ Data augmentation functions loaded


In [15]:
# Dataset class with augmentation
class DimASRDataset(Dataset):
    def __init__(self, data, tokenizer, max_length, augment=False):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.augment = augment
        
        # Flatten data: one sample per aspect
        self.samples = []
        for item in data:
            text = item['Text']
            aspect_list = item.get('Aspect_VA', item.get('Quadruplet', []))
            
            for aspect_item in aspect_list:
                aspect = aspect_item['Aspect']
                va = aspect_item['VA']
                v, a = parse_va(va)
                self.samples.append({
                    'id': item['ID'],
                    'text': text,
                    'aspect': aspect,
                    'valence': normalize_va(v),
                    'arousal': normalize_va(a)
                })
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        text = sample['text']
        aspect = sample['aspect']
        v = sample['valence']
        a = sample['arousal']
        
        # Apply augmentation during training
        if self.augment and random.random() < config.AUGMENTATION_PROB:
            text = augment_text(text, aspect)
            v, a = augment_va(v, a)
        
        # KEEP BERT's two-sequence format (don't change this!)
        encoding = self.tokenizer(
            text,
            aspect,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'valence': torch.tensor(v, dtype=torch.float),
            'arousal': torch.tensor(a, dtype=torch.float),
            'id': sample['id'],
            'text': text,
            'aspect': aspect
        }

# Test dataset (no labels, no augmentation)
class DimASRTestDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        self.samples = []
        for item in data:
            text = item['Text']
            aspects = item.get('Aspect', [])
            
            for aspect in aspects:
                self.samples.append({
                    'id': item['ID'],
                    'text': text,
                    'aspect': aspect
                })
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        encoding = self.tokenizer(
            sample['text'],
            sample['aspect'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'id': sample['id'],
            'aspect': sample['aspect']
        }

print("✓ Dataset classes with augmentation loaded")

✓ Dataset classes with augmentation loaded


In [16]:
# Model with improved regularization
class VAModel(nn.Module):
    def __init__(self, model_name, hidden_dim=256, dropout=0.4):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        
        # IMPROVED: Add dropout to encoder
        self.encoder.config.hidden_dropout_prob = 0.2
        self.encoder.config.attention_probs_dropout_prob = 0.2
        
        encoder_dim = self.encoder.config.hidden_size
        
        # Simple regression head (works best for small data)
        self.regression_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(encoder_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
            nn.Sigmoid()
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output
        va = self.regression_head(pooled)
        return va

print("✓ Model with improved regularization loaded")

✓ Model with improved regularization loaded


In [17]:
# Focal Loss for hard examples
class FocalMSELoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    
    def forward(self, pred, target):
        mse = (pred - target) ** 2
        
        # Weight hard examples (high error) more
        weight = (1 + mse.detach()) ** self.gamma
        
        return (weight * mse).mean()

class CombinedLoss(nn.Module):
    def __init__(self, focal_weight=0.3):
        super().__init__()
        self.focal_weight = focal_weight
        self.mse = nn.MSELoss()
        self.focal = FocalMSELoss(gamma=config.FOCAL_GAMMA)
    
    def forward(self, pred, target):
        mse_loss = self.mse(pred, target)
        focal_loss = self.focal(pred, target)
        
        # Combine losses
        total = (1 - self.focal_weight) * mse_loss + self.focal_weight * focal_loss
        
        return total, {
            'mse': mse_loss.item(),
            'focal': focal_loss.item(),
            'total': total.item()
        }

print("✓ Focal loss loaded")

✓ Focal loss loaded


In [18]:
# Evaluation with comprehensive metrics
def calculate_metrics(predictions, targets):
    pred_v, pred_a = predictions[:, 0], predictions[:, 1]
    target_v, target_a = targets[:, 0], targets[:, 1]
    
    # RMSE - Official metric
    rmse_combined = np.sqrt(np.mean((predictions - targets) ** 2))
    rmse_v = np.sqrt(np.mean((pred_v - target_v) ** 2))
    rmse_a = np.sqrt(np.mean((pred_a - target_a) ** 2))
    
    # MAE
    mae_combined = np.mean(np.abs(predictions - targets))
    
    # Pearson Correlation
    pcc_v, _ = pearsonr(pred_v, target_v)
    pcc_a, _ = pearsonr(pred_a, target_a)
    
    return {
        'rmse': rmse_combined,
        'rmse_v': rmse_v,
        'rmse_a': rmse_a,
        'mae': mae_combined,
        'pcc_v': pcc_v,
        'pcc_a': pcc_a
    }

def evaluate(model, dataloader, device):
    model.eval()
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            valence = batch['valence'].to(device)
            arousal = batch['arousal'].to(device)
            
            va_pred = model(input_ids, attention_mask)
            va_target = torch.stack([valence, arousal], dim=1)
            
            all_predictions.append(va_pred.cpu().numpy())
            all_targets.append(va_target.cpu().numpy())
    
    predictions = np.concatenate(all_predictions, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    
    # Denormalize
    predictions_denorm = denormalize_va(predictions)
    targets_denorm = denormalize_va(targets)
    
    metrics = calculate_metrics(predictions_denorm, targets_denorm)
    
    return metrics

print("✓ Evaluation functions loaded")

✓ Evaluation functions loaded


In [19]:
# Training function
def train_model(model, train_loader, val_loader, device, num_epochs, model_name, log_file):
    # Combined loss (MSE + Focal)
    criterion = CombinedLoss(focal_weight=config.FOCAL_WEIGHT)
    
    optimizer = AdamW(
        model.parameters(),
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY
    )
    
    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(total_steps * config.WARMUP_RATIO)
    
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    
    best_rmse = float('inf')
    patience_counter = 0
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        train_mse = 0
        train_focal = 0
        
        progress_bar = tqdm(train_loader, desc=f'[{model_name}] Epoch {epoch+1}/{num_epochs}')
        
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            valence = batch['valence'].to(device)
            arousal = batch['arousal'].to(device)
            
            va_pred = model(input_ids, attention_mask)
            va_target = torch.stack([valence, arousal], dim=1)
            
            loss, loss_dict = criterion(va_pred, va_target)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
            optimizer.step()
            scheduler.step()
            
            train_loss += loss_dict['total']
            train_mse += loss_dict['mse']
            train_focal += loss_dict['focal']
            
            progress_bar.set_postfix({'loss': f"{loss_dict['total']:.4f}"})
        
        avg_train_loss = train_loss / len(train_loader)
        avg_train_mse = train_mse / len(train_loader)
        avg_train_focal = train_focal / len(train_loader)
        
        # Validation
        val_metrics = evaluate(model, val_loader, device)
        current_lr = scheduler.get_last_lr()[0]
        
        print(f"\n[{model_name}] Epoch {epoch+1}/{num_epochs}")
        print(f"Train - Loss: {avg_train_loss:.4f}, MSE: {avg_train_mse:.4f}, Focal: {avg_train_focal:.4f}")
        print(f"Val - RMSE: {val_metrics['rmse']:.4f} (V: {val_metrics['rmse_v']:.4f}, A: {val_metrics['rmse_a']:.4f})")
        print(f"Val - MAE: {val_metrics['mae']:.4f}, PCC: V={val_metrics['pcc_v']:.3f}, A={val_metrics['pcc_a']:.3f}")
        
        # Log to file
        log_file.write(f"{model_name},{epoch+1},{avg_train_loss:.4f},{avg_train_mse:.4f},{avg_train_focal:.4f},")
        log_file.write(f"{val_metrics['rmse']:.4f},{val_metrics['rmse_v']:.4f},{val_metrics['rmse_a']:.4f},")
        log_file.write(f"{val_metrics['mae']:.4f},{val_metrics['pcc_v']:.4f},{val_metrics['pcc_a']:.4f},{current_lr:.2e}\n")
        log_file.flush()
        
        # Save best model
        if val_metrics['rmse'] < best_rmse:
            best_rmse = val_metrics['rmse']
            patience_counter = 0
            
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'rmse': best_rmse,
                'metrics': val_metrics
            }, f"{config.CHECKPOINT_DIR}/best_model_{model_name}.pt")
            
            print(f"✓ Saved best model (RMSE: {best_rmse:.4f})")
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= config.EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    print(f"\n[{model_name}] Training completed. Best RMSE: {best_rmse:.4f}\n")
    
    return best_rmse

print("✓ Training function loaded")

✓ Training function loaded


In [20]:
# Load data - DOMAIN-SPECIFIC TRAINING
print("="*80)
print("LOADING DATA - DOMAIN-SPECIFIC APPROACH")
print("="*80)

# Load restaurant data
train_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_train_alltasks.jsonl')
val_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_dev_task1.jsonl')

# Filter out dev samples from training
train_rest = [item for item in train_rest if 'dev' not in item['ID']]

print(f"\nRestaurant Domain:")
print(f"  Training: {len(train_rest)} samples")
print(f"  Validation: {len(val_rest)} samples")

# Load laptop data
train_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_train_alltasks.jsonl')
val_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_dev_task1.jsonl')

# Filter out dev samples from training
train_laptop = [item for item in train_laptop if 'dev' not in item['ID']]

print(f"\nLaptop Domain:")
print(f"  Training: {len(train_laptop)} samples")
print(f"  Validation: {len(val_laptop)} samples")

print("\n" + "="*80)

LOADING DATA - DOMAIN-SPECIFIC APPROACH

Restaurant Domain:
  Training: 2113 samples
  Validation: 200 samples

Laptop Domain:
  Training: 3750 samples
  Validation: 200 samples



In [21]:
# Train domain-specific ensemble models
print("="*80)
print("TRAINING ENSEMBLE OF DOMAIN-SPECIFIC MODELS")
print("="*80)

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

# Open master log file
master_log = open(f'{config.LOG_DIR}/training_log.txt', 'w')
master_log.write("Model,Epoch,Train_Loss,Train_MSE,Train_Focal,Val_RMSE,Val_RMSE_V,Val_RMSE_A,Val_MAE,Val_PCC_V,Val_PCC_A,LR\n")

# Store all trained models
all_models = {
    'restaurant': [],
    'laptop': []
}

# Train ensemble for Restaurant domain
print("\n" + "="*80)
print("RESTAURANT DOMAIN - Training Ensemble")
print("="*80)

for seed_idx, seed in enumerate(config.ENSEMBLE_SEEDS):
    print(f"\n{'='*80}")
    print(f"Restaurant Model {seed_idx + 1}/3 (seed={seed})")
    print(f"{'='*80}")
    
    # Set seed
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    # Create datasets with augmentation
    train_dataset = DimASRDataset(train_rest, tokenizer, config.MAX_LENGTH, augment=True)
    val_dataset = DimASRDataset(val_rest, tokenizer, config.MAX_LENGTH, augment=False)
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE)
    
    # Initialize model
    model = VAModel(config.MODEL_NAME, config.HIDDEN_DIM, config.DROPOUT)
    model = model.to(config.DEVICE)
    
    # Train
    model_name = f"rest_seed{seed}"
    best_rmse = train_model(model, train_loader, val_loader, config.DEVICE, 
                           config.EPOCHS, model_name, master_log)
    
    # Store model
    all_models['restaurant'].append({
        'model': model,
        'seed': seed,
        'rmse': best_rmse
    })

# Train ensemble for Laptop domain
print("\n" + "="*80)
print("LAPTOP DOMAIN - Training Ensemble")
print("="*80)

for seed_idx, seed in enumerate(config.ENSEMBLE_SEEDS):
    print(f"\n{'='*80}")
    print(f"Laptop Model {seed_idx + 1}/3 (seed={seed})")
    print(f"{'='*80}")
    
    # Set seed
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    # Create datasets with augmentation
    train_dataset = DimASRDataset(train_laptop, tokenizer, config.MAX_LENGTH, augment=True)
    val_dataset = DimASRDataset(val_laptop, tokenizer, config.MAX_LENGTH, augment=False)
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE)
    
    # Initialize model
    model = VAModel(config.MODEL_NAME, config.HIDDEN_DIM, config.DROPOUT)
    model = model.to(config.DEVICE)
    
    # Train
    model_name = f"laptop_seed{seed}"
    best_rmse = train_model(model, train_loader, val_loader, config.DEVICE, 
                           config.EPOCHS, model_name, master_log)
    
    # Store model
    all_models['laptop'].append({
        'model': model,
        'seed': seed,
        'rmse': best_rmse
    })

master_log.close()

print("\n" + "="*80)
print("TRAINING COMPLETED!")
print("="*80)
print(f"\nRestaurant models trained: {len(all_models['restaurant'])}")
print(f"Laptop models trained: {len(all_models['laptop'])}")
print(f"\nTraining log: {config.LOG_DIR}/training_log.txt")

TRAINING ENSEMBLE OF DOMAIN-SPECIFIC MODELS


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...


[laptop_seed456] Epoch 10/30
Train - Loss: 0.0062, MSE: 0.0061, Focal: 0.0065
Val - RMSE: 0.8340 (V: 0.8115, A: 0.8559)
Val - MAE: 0.6118, PCC: V=0.878, A=0.501


[laptop_seed456] Epoch 11/30:   0%|          | 0/167 [00:00<?, ?it/s]


[laptop_seed456] Epoch 11/30
Train - Loss: 0.0061, MSE: 0.0059, Focal: 0.0064
Val - RMSE: 0.8107 (V: 0.8340, A: 0.7867)
Val - MAE: 0.6051, PCC: V=0.868, A=0.501


[laptop_seed456] Epoch 12/30:   0%|          | 0/167 [00:00<?, ?it/s]


[laptop_seed456] Epoch 12/30
Train - Loss: 0.0057, MSE: 0.0056, Focal: 0.0061
Val - RMSE: 0.8051 (V: 0.7935, A: 0.8164)
Val - MAE: 0.5913, PCC: V=0.882, A=0.510

Early stopping at epoch 12

[laptop_seed456] Training completed. Best RMSE: 0.7820


TRAINING COMPLETED!

Restaurant models trained: 3
Laptop models trained: 3

Training log: ./subtask1_logs/training_log.txt


In [22]:
# Load best models for each domain
print("\n" + "="*80)
print("LOADING BEST MODELS FOR INFERENCE")
print("="*80)

# Load best restaurant models
restaurant_models = []
for seed in config.ENSEMBLE_SEEDS:
    model = VAModel(config.MODEL_NAME, config.HIDDEN_DIM, config.DROPOUT)
    checkpoint = torch.load(f"{config.CHECKPOINT_DIR}/best_model_rest_seed{seed}.pt", weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(config.DEVICE)
    model.eval()
    restaurant_models.append(model)
    print(f"✓ Loaded Restaurant model (seed={seed}), RMSE={checkpoint['rmse']:.4f}")

# Load best laptop models
laptop_models = []
for seed in config.ENSEMBLE_SEEDS:
    model = VAModel(config.MODEL_NAME, config.HIDDEN_DIM, config.DROPOUT)
    checkpoint = torch.load(f"{config.CHECKPOINT_DIR}/best_model_laptop_seed{seed}.pt", weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(config.DEVICE)
    model.eval()
    laptop_models.append(model)
    print(f"✓ Loaded Laptop model (seed={seed}), RMSE={checkpoint['rmse']:.4f}")

print(f"\nTotal models loaded: {len(restaurant_models) + len(laptop_models)}")


LOADING BEST MODELS FOR INFERENCE
✓ Loaded Restaurant model (seed=42), RMSE=0.8447
✓ Loaded Restaurant model (seed=123), RMSE=0.8454
✓ Loaded Restaurant model (seed=456), RMSE=0.7739
✓ Loaded Laptop model (seed=42), RMSE=0.7838
✓ Loaded Laptop model (seed=123), RMSE=0.8182
✓ Loaded Laptop model (seed=456), RMSE=0.7820

Total models loaded: 6


In [23]:
# Ensemble prediction function
def predict_ensemble(models, dataloader, device):
    """Predict using ensemble of models"""
    all_predictions = []
    
    # Get predictions from each model
    for model in models:
        model.eval()
        model_preds = []
        
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                
                va_pred = model(input_ids, attention_mask)
                model_preds.append(va_pred.cpu().numpy())
        
        all_predictions.append(np.concatenate(model_preds, axis=0))
    
    # Average predictions across ensemble
    ensemble_pred = np.mean(all_predictions, axis=0)
    
    return ensemble_pred

def format_predictions(ensemble_pred, dataloader):
    """Format predictions into output structure"""
    predictions = {}
    
    # Denormalize
    v_pred = denormalize_va(ensemble_pred[:, 0])
    a_pred = denormalize_va(ensemble_pred[:, 1])
    
    # Get IDs and aspects from dataloader
    idx = 0
    for batch in dataloader:
        ids = batch['id']
        aspects = batch['aspect']
        batch_size = len(ids)
        
        for i in range(batch_size):
            id_ = ids[i]
            aspect = aspects[i]
            v = v_pred[idx]
            a = a_pred[idx]
            
            if id_ not in predictions:
                predictions[id_] = []
            
            predictions[id_].append({
                'Aspect': aspect,
                'VA': format_va(v, a)
            })
            
            idx += 1
    
    return predictions

def save_predictions(predictions, original_data, output_path):
    """Save predictions in JSONL format"""
    with open(output_path, 'w', encoding='utf-8') as f:
        for item in original_data:
            id_ = item['ID']
            output = {
                'ID': id_,
                'Aspect_VA': predictions.get(id_, [])
            }
            f.write(json.dumps(output) + '\n')
    print(f"✓ Predictions saved to {output_path}")

print("✓ Ensemble prediction functions loaded")

✓ Ensemble prediction functions loaded


In [24]:
# Generate predictions for test sets
print("\n" + "="*80)
print("GENERATING ENSEMBLE PREDICTIONS FOR TEST SETS")
print("="*80)

# Restaurant test
print("\nRestaurant test set...")
test_rest_data = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_test_task1.jsonl')
test_rest_dataset = DimASRTestDataset(test_rest_data, tokenizer, config.MAX_LENGTH)
test_rest_loader = DataLoader(test_rest_dataset, batch_size=config.BATCH_SIZE)

print("Predicting with ensemble...")
ensemble_pred_rest = predict_ensemble(restaurant_models, test_rest_loader, config.DEVICE)
predictions_rest = format_predictions(ensemble_pred_rest, test_rest_loader)
save_predictions(predictions_rest, test_rest_data, 
                f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl')

# Laptop test
print("\nLaptop test set...")
test_laptop_data = load_jsonl(f'{config.DATA_DIR}/eng_laptop_test_task1.jsonl')
test_laptop_dataset = DimASRTestDataset(test_laptop_data, tokenizer, config.MAX_LENGTH)
test_laptop_loader = DataLoader(test_laptop_dataset, batch_size=config.BATCH_SIZE)

print("Predicting with ensemble...")
ensemble_pred_laptop = predict_ensemble(laptop_models, test_laptop_loader, config.DEVICE)
predictions_laptop = format_predictions(ensemble_pred_laptop, test_laptop_loader)
save_predictions(predictions_laptop, test_laptop_data, 
                f'{config.OUTPUT_DIR}/laptop_test_predictions.jsonl')

print("\n" + "="*80)
print("ALL PREDICTIONS COMPLETED!")
print("="*80)


GENERATING ENSEMBLE PREDICTIONS FOR TEST SETS

Restaurant test set...
Predicting with ensemble...
✓ Predictions saved to ./subtask1_outputs/restaurant_test_predictions.jsonl

Laptop test set...
Predicting with ensemble...
✓ Predictions saved to ./subtask1_outputs/laptop_test_predictions.jsonl

ALL PREDICTIONS COMPLETED!


In [25]:
# Display sample predictions
print("\n" + "="*80)
print("SAMPLE PREDICTIONS")
print("="*80)

print("\nRestaurant samples:")
with open(f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            pred = json.loads(line)
            print(f"\n--- Sample {i+1} ---")
            print(f"Text: {test_rest_data[i]['Text']}")
            print(f"ID: {pred['ID']}")
            for aspect_va in pred['Aspect_VA']:
                print(f"  → Aspect: {aspect_va['Aspect']}, VA: {aspect_va['VA']}")
            print("-" * 50)

print("\nLaptop samples:")
with open(f'{config.OUTPUT_DIR}/laptop_test_predictions.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i < 3:
            pred = json.loads(line)
            print(f"\n--- Sample {i+1} ---")
            print(f"Text: {test_laptop_data[i]['Text']}")
            print(f"ID: {pred['ID']}")
            for aspect_va in pred['Aspect_VA']:
                print(f"  → Aspect: {aspect_va['Aspect']}, VA: {aspect_va['VA']}")
            print("-" * 50)

print("\n" + "="*80)


SAMPLE PREDICTIONS

Restaurant samples:

--- Sample 1 ---
Text: A friend suggested this cafe for a lunch date a while back and I cannot stay away
ID: rest26_aspect_va_test_1
  → Aspect: cafe, VA: 5.81#6.20
--------------------------------------------------

--- Sample 2 ---
Text: The beer selection is second to none , but this time we had one drink and decided to spend our money elsewhere for dinner
ID: rest26_aspect_va_test_2
  → Aspect: beer selection, VA: 5.10#5.50
--------------------------------------------------

--- Sample 3 ---
Text: It was pretty bland for my liking - in complete contrast to the sausage and pepper pizza - and the pepperoni was pretty sparse
ID: rest26_aspect_va_test_3
  → Aspect: pepper pizza, VA: 4.18#5.72
  → Aspect: pepperoni, VA: 4.25#5.66
  → Aspect: sausage, VA: 4.14#5.73
--------------------------------------------------

--- Sample 4 ---
Text: Half price beer during a generous happy hour , a huge deck that welcomes dogs , and delicious key lime pie
ID